# Despliegue Streamlit Vinos

- Cargamos el modelo
- Capturamos los datos del vino con la interfaz gráfica
- Preparar los datos: dummies, normalizar
- Aplicamos el modelo para la predicción

La aplicación se ejecuta con `streamlit run despliegue_streamlit_vinos_doa_ftm.py` (este notebook exportado a .py). En el notebook los controles toman su valor inicial, por eso las celdas de preparación y la verificación final muestran la predicción para la mediana de un vino blanco.

In [ ]:
# Cargamos librerías principales
import pickle
import pandas as pd
import streamlit as st

NOMBRE_APP = 'Vinalia'

In [ ]:
# Cargamos el modelo
filename = 'modelo-cla-vinos-hiper.pkl' # Red neuronal hiperparametrizada (Hiperparametrizacion_GridSearch_Vinos_DOA_FTM.ipynb)
with open(filename, 'rb') as f:
    modelo, labelencoder, variables, min_max_scaler = pickle.load(f)

In [ ]:
# Resultados de la red neuronal hiperparametrizada con GridSearch (Hiperparametrizacion_GridSearch_Vinos_DOA_FTM.ipynb)
# StratifiedKFold(n_splits=10, shuffle=True, random_state=42), promedio de los 10 folds, scoring f1_macro
METRICAS = {
    'f1_test': 0.749,   # grid.best_score_ / mean_test_score
    'f1_train': 0.768,  # mean_train_score
    'f1_vc': 0.739,     # red neuronal antes de hiperparametrizar (Validación_Cruzada_Vinos_DOA_FTM.ipynb)
}
# F1 macro de la validación cruzada (Validación_Cruzada_Vinos_DOA_FTM.ipynb) + red neuronal tras GridSearch
COMPARACION_F1 = {'Red neuronal': 0.749, 'SVM': 0.736, 'XGBoost': 0.734, 'Random Forest': 0.721,
                  'KNN': 0.718, 'Árbol': 0.704}
MODELO_ELEGIDO = 'Red neuronal'

# Datos de entrenamiento (wine_data.csv)
N_VINOS, N_BLANCOS, N_TINTOS, N_BUENOS = 4979, 3961, 1018, 3159
LINEA_BASE = N_BUENOS / N_VINOS  # proporción de vinos de calidad Buena

# Correlación de Pearson con calidad = Buena (Seleccion_Factores_Vinos_DOA_FTM.ipynb)
CORRELACIONES = [('Alcohol', 0.41), ('Densidad', -0.29), ('Acidez volátil', -0.26),
                 ('Cloruros', -0.18), ('Tipo tinto', -0.10)]

# Variables del formulario: (columna, etiqueta, unidad, mín, máx, paso, formato)
# Mín y máx son los observados en los datos de entrenamiento
GRUPOS = {
    'Acidez y composición': [
        ('fixed acidity', 'Acidez fija', 'g/L', 3.8, 15.9, 0.1, '%.1f'),
        ('volatile acidity', 'Acidez volátil', 'g/L', 0.08, 1.58, 0.01, '%.2f'),
        ('citric acid', 'Ácido cítrico', 'g/L', 0.0, 1.66, 0.01, '%.2f'),
        ('residual sugar', 'Azúcar residual', 'g/L', 0.6, 65.8, 0.1, '%.1f'),
        ('chlorides', 'Cloruros', 'g/L', 0.009, 0.611, 0.001, '%.3f'),
        ('pH', 'pH', '', 2.72, 4.01, 0.01, '%.2f'),
    ],
    'Sulfitos, densidad y alcohol': [
        ('free sulfur dioxide', 'SO₂ libre', 'mg/L', 1.0, 289.0, 1.0, '%.0f'),
        ('total sulfur dioxide', 'SO₂ total', 'mg/L', 6.0, 440.0, 1.0, '%.0f'),
        ('density', 'Densidad', 'g/cm³', 0.9871, 1.0390, 0.0001, '%.4f'),
        ('sulphates', 'Sulfatos', 'g/L', 0.22, 2.0, 0.01, '%.2f'),
        ('alcohol', 'Alcohol', '% vol', 8.0, 14.9, 0.1, '%.1f'),
    ],
}

# Valores iniciales: mediana de cada tipo de vino en los datos de entrenamiento
MEDIANAS = {
    'blanco': {'fixed acidity': 6.8, 'volatile acidity': 0.26, 'citric acid': 0.32, 'residual sugar': 4.7,
               'chlorides': 0.042, 'free sulfur dioxide': 33.0, 'total sulfur dioxide': 133.0,
               'density': 0.9935, 'pH': 3.18, 'sulphates': 0.48, 'alcohol': 10.4},
    'tinto': {'fixed acidity': 7.9, 'volatile acidity': 0.52, 'citric acid': 0.25, 'residual sugar': 2.2,
              'chlorides': 0.079, 'free sulfur dioxide': 13.0, 'total sulfur dioxide': 38.0,
              'density': 0.9967, 'pH': 3.31, 'sulphates': 0.62, 'alcohol': 10.2},
}


def miles(n):
    return f'{n:,}'.replace(',', '.')  # separador de miles con punto: 4.979

# Interfaz gráfica

In [ ]:
# Se crea interfaz gráfica con streamlit para captura de los datos
st.set_page_config(page_title=f'{NOMBRE_APP} · Calidad de vinos', page_icon='🍷', layout='wide')

st.markdown("""
    <style>
    @import url('https://fonts.googleapis.com/css2?family=Manrope:wght@400;500;600;700;800&display=swap');

    :root {
        --vino: #7a1f3d; --vino-osc: #5e1830; --vino-claro: #f6e8ec;
        --texto: #1f1a1c; --suave: #6b6266; --borde: #e7e1dd;
        --fondo: #faf8f6; --tarjeta: #ffffff;
        --bueno: #2f6b4f; --malo: #a12d3a;
    }

    /* Base */
    .stApp { background: var(--fondo); }
    .stApp, .stApp *:not([data-testid="stIconMaterial"]):not(.material-symbols-rounded):not(code) {
        font-family: 'Manrope', system-ui, -apple-system, sans-serif;
    }
    [data-testid="stHeader"] { background: transparent; }
    /* Oculta el ícono de enlace que Streamlit agrega a los títulos */
    [data-testid="stHeaderActionElements"] { display: none !important; }
    [data-testid="stMain"], [data-testid="stAppViewContainer"] { scroll-behavior: smooth; }
    .block-container, [data-testid="stMainBlockContainer"] { max-width: 1240px; padding: 1.25rem 2rem 2rem; }

    /* Barra superior */
    .nav { display: flex; align-items: center; justify-content: space-between; padding: .5rem 0 1.25rem; }
    .nav .marca { font-weight: 800; font-size: 1.1rem; color: var(--texto); letter-spacing: -.01em; }
    .nav .links a { color: var(--suave); text-decoration: none; font-size: .9rem; font-weight: 500; margin-left: 1.5rem; }
    .nav .links a:hover { color: var(--vino); }

    /* Portada */
    .hero {
        display: grid; grid-template-columns: 1.5fr 1fr; align-items: center; gap: 2rem;
        background: radial-gradient(circle at 85% 30%, #9b3157 0%, transparent 55%),
                    linear-gradient(135deg, #4a1226 0%, #6e1c38 55%, #7a1f3d 100%);
        border-radius: 24px; padding: 3.5rem; color: #fff;
        box-shadow: 0 20px 50px rgba(74,18,38,.22);
    }
    .hero .eyebrow {
        display: inline-block; font-size: .75rem; font-weight: 600; letter-spacing: .08em; text-transform: uppercase;
        color: #f6dbe3; background: rgba(255,255,255,.1); border: 1px solid rgba(255,255,255,.18);
        border-radius: 99px; padding: .35rem .8rem;
    }
    .hero h1 {
        font-size: 2.8rem; font-weight: 800; line-height: 1.12; letter-spacing: -.03em;
        color: #fff; margin: 1rem 0 .9rem; padding: 0;
    }
    .hero h1 em { font-style: normal; color: #f3b9c9; }
    .hero .sub { font-size: 1.05rem; line-height: 1.6; color: #f3dfe5; max-width: 580px; }
    .hero .cta {
        display: inline-block; margin-top: 1.75rem; background: #fff; color: var(--vino) !important;
        font-weight: 700; text-decoration: none; padding: .85rem 1.5rem; border-radius: 10px;
        box-shadow: 0 8px 20px rgba(0,0,0,.18); transition: transform .15s ease;
    }
    .hero .cta:hover { transform: translateY(-2px); }
    .hero .stats { display: flex; gap: 2.5rem; margin-top: 2.25rem; padding-top: 1.5rem; border-top: 1px solid rgba(255,255,255,.15); }
    .hero .stats b { display: block; font-size: 1.6rem; font-weight: 800; color: #fff; font-variant-numeric: tabular-nums; }
    .hero .stats span { font-size: .82rem; color: #e9c6d0; }
    .hero .visual { display: flex; justify-content: center; }
    .hero .visual svg { width: 100%; max-width: 260px; height: auto; }

    /* Encabezados de sección */
    .titulo-seccion { text-align: center; margin: 4rem 0 2rem; }
    .titulo-seccion .eyebrow { font-size: .78rem; font-weight: 700; letter-spacing: .08em; text-transform: uppercase; color: var(--vino); }
    .titulo-seccion h2 { font-size: 2.1rem; font-weight: 800; letter-spacing: -.02em; color: var(--texto); margin: .4rem 0 .5rem; padding: 0; }
    .titulo-seccion p { color: var(--suave); font-size: 1rem; line-height: 1.6; max-width: 700px; margin: 0 auto; }

    /* Tarjetas del formulario, la muestra y la predicción: la columna derecha mide lo mismo que el formulario */
    .st-key-formulario, .st-key-panel, .st-key-tarjeta-prediccion {
        background: var(--tarjeta); border: 1px solid var(--borde); border-radius: 18px;
        padding: 1.75rem; box-shadow: 0 1px 3px rgba(31,26,28,.05);
    }
    [data-testid="stColumn"]:has(.st-key-formulario), [data-testid="stColumn"]:has(.st-key-panel) {
        align-self: stretch; display: flex; flex-direction: column;
    }
    [data-testid="stColumn"]:has(.st-key-formulario) > [data-testid="stVerticalBlock"],
    [data-testid="stColumn"]:has(.st-key-panel) > [data-testid="stVerticalBlock"],
    [data-testid="stLayoutWrapper"]:has(> .st-key-formulario), [data-testid="stLayoutWrapper"]:has(> .st-key-panel),
    .st-key-formulario, .st-key-panel { flex: 1 1 auto; }
    .grupo { font-size: .95rem; font-weight: 700; color: var(--texto); margin: .75rem 0 .25rem; }
    .grupo.sliders { margin-top: 0; margin-bottom: .9rem; }
    .panel-titulo { font-size: 1.15rem; font-weight: 700; color: var(--texto); }
    .st-key-formulario .panel-titulo { margin-bottom: .35rem; }

    /* Sliders: mín y máx siempre visibles (Streamlit solo los muestra al pasar el mouse) */
    [data-testid="stSliderTickBar"] { display: none !important; }
    .rango {
        display: flex; justify-content: space-between; margin: -2.1rem 0 .2rem; padding: 0 .15rem;
        font-size: .75rem; color: var(--suave); font-variant-numeric: tabular-nums;
    }

    /* Tabla de la muestra */
    .tabla { width: 100%; border: none; border-collapse: collapse; margin-top: .5rem; font-size: .86rem; font-variant-numeric: tabular-nums; }
    /* Cabecera y filas con la misma altura */
    .tabla tr { border: none; }
    .tabla th, .tabla td { border: none; background: none; height: 2.65rem; padding-top: 0; padding-bottom: 0; vertical-align: middle; }
    .tabla th {
        text-align: right; font-size: .78rem; font-weight: 700; letter-spacing: .04em; text-transform: uppercase;
        color: var(--suave); padding-left: .5rem; padding-right: .5rem; border-bottom: 1px solid var(--borde);
    }
    .tabla th:first-child, .tabla td:first-child { text-align: left; padding-left: 0; }
    .tabla td { text-align: right; padding-left: .5rem; padding-right: .5rem; border-bottom: 1px solid var(--borde); color: var(--texto); }
    .tabla td:last-child, .tabla th:last-child { padding-right: 0; }
    .tabla td.ref { color: var(--suave); }
    .tabla tbody tr:last-child td { border-bottom: none; }
    .tabla td .und { font-size: .72rem; color: var(--suave); margin-left: .15rem; }

    /* Resultado: altura fija para que no cambie el alto de las tarjetas */
    /* Streamlit da margin-bottom negativo al markdown: se anula para que el espacio inferior iguale al superior */
    .st-key-tarjeta-prediccion [data-testid="stMarkdownContainer"] { margin-bottom: 0; }
    .resultado { display: flex; flex-direction: column; gap: .9rem; margin-top: .6rem; }
    .resultado.buena { --c: var(--bueno); }
    .resultado.mala { --c: var(--malo); }
    .resultado .fila { display: flex; align-items: center; justify-content: space-between; }
    .resultado .valor { font-size: 1.6rem; font-weight: 800; letter-spacing: -.02em; color: var(--texto); line-height: 1.15; }
    .resultado .prob { font-size: 1.6rem; font-weight: 800; color: var(--c); line-height: 1.15; font-variant-numeric: tabular-nums; }
    .resultado .barra { position: relative; height: 6px; background: var(--borde); border-radius: 99px; }
    .resultado .barra span { display: block; height: 100%; background: var(--c); border-radius: 99px; transition: width .25s ease; }
    .resultado .meta { font-size: .8rem; color: var(--suave); }

    /* Cómo funciona */
    .pasos { display: grid; grid-template-columns: repeat(4, 1fr); gap: 1rem; }
    .paso { background: var(--tarjeta); border: 1px solid var(--borde); border-radius: 18px; padding: 1.5rem; }
    .paso .n {
        display: inline-flex; align-items: center; justify-content: center; width: 2.2rem; height: 2.2rem;
        border-radius: 10px; background: var(--vino-claro); color: var(--vino); font-weight: 800;
    }
    .paso h3 { font-size: 1.15rem; font-weight: 700; color: var(--texto); margin: .9rem 0 .4rem; padding: 0; }
    .paso p { font-size: .9rem; color: var(--suave); line-height: 1.55; margin: 0; }
    .paso code {
        font-family: ui-monospace, SFMono-Regular, Menlo, monospace; font-size: .78rem;
        background: var(--fondo); border: 1px solid var(--borde); border-radius: 5px; padding: .05rem .3rem; color: var(--texto);
    }

    /* Banda de métricas */
    .banda {
        display: grid; grid-template-columns: repeat(4, 1fr); gap: 1rem;
        background: var(--vino-claro); border-radius: 18px; padding: 1.75rem 2rem; text-align: center;
    }
    .banda b { display: block; font-size: 2rem; font-weight: 800; color: var(--vino); letter-spacing: -.02em; font-variant-numeric: tabular-nums; }
    .banda span { display: block; font-size: .95rem; font-weight: 600; color: var(--texto); }
    .banda small { display: block; font-size: .86rem; color: var(--suave); margin-top: .15rem; }

    /* Gráficos de barras */
    .graficos { display: grid; grid-template-columns: 1fr 1fr; gap: 1rem; margin-top: 1rem; }
    .grafico { background: var(--tarjeta); border: 1px solid var(--borde); border-radius: 18px; padding: 1.5rem; }
    .grafico h3 { font-size: 1.15rem; font-weight: 700; color: var(--texto); margin: 0 0 .45rem; padding: 0; }
    .grafico .nota { font-size: .86rem; color: var(--suave); margin-bottom: 1rem; }
    .fila-barra { display: grid; grid-template-columns: 7.5rem 1fr 3rem; align-items: center; gap: .75rem; margin: .6rem 0; font-size: .9rem; color: var(--texto); }
    .fila-barra .num { text-align: right; font-variant-numeric: tabular-nums; }
    .fila-barra .pista { position: relative; height: 10px; }
    .fila-barra .pista span { position: absolute; top: 0; height: 100%; border-radius: 99px; }
    .fila-barra .pista span.base { left: 0; background: #d8b8c3; }
    .fila-barra.activa { font-weight: 700; }
    .fila-barra.activa .pista span.base { background: var(--vino); }
    .fila-barra .pista.centro::before { content: ''; position: absolute; left: 50%; top: -3px; bottom: -3px; width: 1px; background: var(--borde); }
    .fila-barra .pista span.pos { left: 50%; background: var(--bueno); }
    .fila-barra .pista span.neg { right: 50%; background: var(--malo); }
    .aviso { font-size: .9rem; color: var(--suave); line-height: 1.6; margin: 1.25rem auto 0; max-width: 820px; text-align: center; }

    /* Pie */
    .pie { border-top: 1px solid var(--borde); margin-top: 3.5rem; padding-top: 1.25rem; text-align: center; font-size: .86rem; color: var(--suave); }

    /* Pantallas pequeñas */
    @media (max-width: 900px) {
        .hero { grid-template-columns: 1fr; padding: 2.25rem 1.5rem; }
        .hero h1 { font-size: 2.1rem; }
        .hero .stats { gap: 1.5rem; flex-wrap: wrap; }
        .hero .visual { display: none; }
        .pasos, .banda { grid-template-columns: 1fr 1fr; }
        .graficos { grid-template-columns: 1fr; }
        .nav .links { display: none; }
    }
    </style>
""", unsafe_allow_html=True)

In [ ]:
# Barra superior y portada
st.markdown(f"""
<div class="nav">
<div class="marca">🍷 {NOMBRE_APP}</div>
<div class="links"><a href="#prediccion">Predicción</a><a href="#como-funciona">Cómo funciona</a><a href="#modelo">Evaluación</a></div>
</div>
<div class="hero">
<div>
<div class="eyebrow">Clasificación de la calidad de vinos - Red neuronal MLP</div>
<h1>La calidad de un vino, <em>a partir de sus números</em></h1>
<div class="sub">Clasificador entrenado con {miles(N_VINOS)} vinos Vinho Verde (joven), {miles(N_BLANCOS)} blancos y {miles(N_TINTOS)} tintos,
a partir de 11 mediciones fisicoquímicas. Ingresa el análisis de laboratorio de un vino y obtén la probabilidad
de que sea de calidad buena.</div>
<a class="cta" href="#prediccion">Hacer una predicción</a>
<div class="stats">
<div><b>{METRICAS['f1_test']:.3f}</b><span>F1 macro en validación</span></div>
<div><b>+{METRICAS['f1_test'] - METRICAS['f1_vc']:.3f}</b><span>F1 con GridSearch</span></div>
<div><b>{(METRICAS['f1_train'] - METRICAS['f1_test']) * 100:.1f} pp</b><span>brecha train–test</span></div>
</div>
</div>
<div class="visual">
<svg viewBox="0 0 200 250" xmlns="http://www.w3.org/2000/svg" aria-hidden="true">
<circle cx="100" cy="115" r="95" fill="rgba(255,255,255,0.07)"/>
<path d="M55 35 H145 C145 100 130 140 100 145 C70 140 55 100 55 35 Z" fill="rgba(255,255,255,0.10)" stroke="rgba(255,255,255,0.65)" stroke-width="2.5"/>
<path d="M58 78 H142 C139 115 125 138 100 142 C75 138 61 115 58 78 Z" fill="#c2416a"/>
<ellipse cx="100" cy="78" rx="42" ry="5" fill="#d9577f"/>
<path d="M70 48 C68 78 72 105 83 125" stroke="rgba(255,255,255,0.35)" stroke-width="4" fill="none" stroke-linecap="round"/>
<rect x="97" y="145" width="6" height="70" rx="3" fill="rgba(255,255,255,0.65)"/>
<ellipse cx="100" cy="218" rx="38" ry="6" fill="rgba(255,255,255,0.65)"/>
</svg>
</div>
</div>
""", unsafe_allow_html=True)

In [ ]:
# Sección de predicción
st.markdown("""
<div id="prediccion" class="titulo-seccion">
<div class="eyebrow">Predicción</div>
<h2>Análisis del vino</h2>
<p>Los valores iniciales son la mediana del tipo de vino seleccionado. Los extremos de cada control son el mínimo
y el máximo observados en el entrenamiento: fuera de ese rango el modelo estaría extrapolando.</p>
</div>
""", unsafe_allow_html=True)

col_form, col_panel = st.columns([1.5, 1], gap='large')

# Captura de datos
valores = {}
with col_form:
    with st.container(key='formulario'):
        st.markdown('<div class="panel-titulo">Valores del vino</div>', unsafe_allow_html=True)

        # Tipo de vino
        st.markdown('<div class="grupo">Tipo de vino</div>', unsafe_allow_html=True)
        tipo = st.radio('Tipo de vino', ['blanco', 'tinto'], format_func=str.capitalize, horizontal=True,
                        label_visibility='collapsed')
        medianas = MEDIANAS[tipo]

        for col_grupo, (grupo, campos) in zip(st.columns(2, gap='large'), GRUPOS.items()):
            with col_grupo:
                st.markdown(f'<div class="grupo sliders">{grupo}</div>', unsafe_allow_html=True)
                for columna, etiqueta, unidad, minimo, maximo, paso, formato in campos:
                    # La key incluye el tipo para que al cambiarlo se carguen sus medianas
                    valores[columna] = st.slider(
                        f'{etiqueta} ({unidad})' if unidad else etiqueta,
                        min_value=minimo, max_value=maximo, value=medianas[columna], step=paso,
                        format=formato, key=f'{columna}-{tipo}',
                        help=f'Variable `{columna}`. Mediana en vinos {tipo}s: {formato % medianas[columna]}')
                    st.markdown(f'<div class="rango"><span>{formato % minimo}</span>'
                                f'<span>{formato % maximo}</span></div>', unsafe_allow_html=True)

# Preparación de los datos capturados

In [ ]:
# Dataframe
predictoras_numericas = ['fixed acidity', 'volatile acidity', 'citric acid',
                         'residual sugar', 'chlorides', 'free sulfur dioxide',
                         'total sulfur dioxide', 'density', 'pH', 'sulphates',
                         'alcohol']

datos = [[valores[c] for c in predictoras_numericas] + [tipo]]
data = pd.DataFrame(datos, columns=predictoras_numericas + ['tipo']) #Dataframe con los mismos nombres de variables

In [ ]:
# Se realiza la preparación de datos
data_preparada=data.copy()

# En despliegue drop_first= False
data_preparada = pd.get_dummies(data_preparada, columns=['tipo'], drop_first=False, dtype=int)

In [ ]:
# Se adicionan las columnas faltantes (queda solo tipo_tinto, igual que en el entrenamiento)
data_preparada=data_preparada.reindex(columns=variables,fill_value=0)

In [ ]:
# Se normaliza para predecir con Knn, Red, SVM, Reg
# En los despliegues no se llama fit
# Como el modelo final es una Red Neuronal SÍ se normaliza
data_preparada[predictoras_numericas]= min_max_scaler.transform(data_preparada[predictoras_numericas])

# Predicción

In [ ]:
# Tabla de la muestra: valor ingresado frente a la mediana del tipo
filas = ''
for campos in GRUPOS.values():
    for columna, etiqueta, unidad, minimo, maximo, paso, formato in campos:
        valor, mediana = valores[columna], medianas[columna]
        filas += (f'<tr><td>{etiqueta}</td>'
                  f'<td>{formato % valor}<span class="und">{unidad}</span></td>'
                  f'<td class="ref">{formato % mediana}</td></tr>')

# Columna derecha: tarjeta de la muestra y tarjeta de la predicción
with col_panel:
    with st.container(key='panel'):
        st.markdown('<div class="panel-titulo">Muestra a evaluar</div>', unsafe_allow_html=True)
        st.markdown(f'<table class="tabla"><thead><tr><th>Variable</th><th>Valor</th><th>Mediana</th></tr></thead>'
                    f'<tbody>{filas}</tbody></table>', unsafe_allow_html=True)

    with st.container(key='tarjeta-prediccion'):
        # Predicción: se recalcula en cada cambio de los controles (Streamlit vuelve a ejecutar el script)

        # Hacemos la predicción con la Red Neuronal
        Y_pred = modelo.predict(data_preparada)

        # Se convierte 0/1 a la etiqueta original (Buena/Mala)
        Y_pred_etiqueta = labelencoder.inverse_transform(Y_pred)

        # Probabilidad de la clase Buena
        idx_buena = list(modelo.classes_).index(labelencoder.transform(['Buena'])[0])
        p_buena = modelo.predict_proba(data_preparada)[0][idx_buena]

        if Y_pred_etiqueta[0] == 'Buena':
            clase, titulo = 'buena', 'Buena'
        else:
            clase, titulo = 'mala', 'Mala'
        # El porcentaje es la probabilidad de clase Buena; con más de 50 % se clasifica como Buena
        if abs(p_buena - 0.5) < 0.1:
            explicacion = 'Cerca del 50 %: el modelo no está seguro, resultado poco concluyente'
        else:
            explicacion = 'Probabilidad de que los catadores le den una nota de 6 o más'

        st.markdown(f"""
<div class="panel-titulo">Predicción</div>
<div class="resultado {clase}">
<div class="fila"><div class="valor">{titulo}</div><div class="prob">{p_buena:.1%}</div></div>
<div class="barra"><span style="width:{p_buena:.1%}"></span></div>
<div class="meta">{explicacion}</div>
</div>
""", unsafe_allow_html=True)

# Información del modelo

In [ ]:
# Cómo funciona: parámetros leídos del modelo cargado
params = modelo.get_params()
capas = params['hidden_layer_sizes']
capas = list(capas) if isinstance(capas, tuple) else [capas]
# Ej.: (100, 50) -> "2 capas ocultas de 100 y 50 neuronas"
neuronas = ', '.join(str(n) for n in capas[:-1]) + ' y ' + str(capas[-1]) if len(capas) > 1 else str(capas[0])
capas = f'{len(capas)} capa{"s" if len(capas) > 1 else ""} oculta{"s" if len(capas) > 1 else ""} de {neuronas} neuronas'

st.markdown(f"""
<div id="como-funciona" class="titulo-seccion">
<div class="eyebrow">Cómo funciona</div>
<h2>Del análisis de laboratorio a la predicción</h2>
<p>El mismo preprocesamiento del entrenamiento se aplica a cada muestra antes de pasarla a la red.</p>
</div>
<div class="pasos">
<div class="paso"><span class="n">1</span><h3>Variable objetivo</h3><p>La nota de los catadores (escala 0–10)
se binarizó: <b>Buena</b> si es ≥ 6 ({LINEA_BASE:.1%} de los vinos) y <b>Mala</b> si es menor.</p></div>
<div class="paso"><span class="n">2</span><h3>Preprocesamiento</h3><p>El tipo se codifica como dummy <code>tipo_tinto</code>
y las 11 variables numéricas se escalan a [0, 1] con el <code>MinMaxScaler</code> ajustado en entrenamiento (aquí solo
<code>transform</code>).</p></div>
<div class="paso"><span class="n">3</span><h3>Red neuronal</h3><p><code>MLPClassifier</code> con
{capas}, activación <code>{params['activation']}</code>, solver <code>{params['solver']}</code> y tasa de aprendizaje
{params['learning_rate_init']}.</p></div>
<div class="paso"><span class="n">4</span><h3>Salida</h3><p><code>predict_proba</code> entrega P(Buena); la muestra
se clasifica como Buena si supera 0.5 y <code>LabelEncoder</code> devuelve la etiqueta original.</p></div>
</div>
""", unsafe_allow_html=True)

In [ ]:
# Recordar medida de desempeño del modelo (validación cruzada)
brecha = (METRICAS['f1_train'] - METRICAS['f1_test']) * 100
max_f1 = max(COMPARACION_F1.values())
barras_modelos = ''.join(
    f'<div class="fila-barra{" activa" if nombre == MODELO_ELEGIDO else ""}"><span>{nombre}</span>'
    f'<div class="pista"><span class="base" style="width:{f1 / max_f1:.0%}"></span></div>'
    f'<span class="num">{f1:.3f}</span></div>'
    for nombre, f1 in COMPARACION_F1.items())
barras_corr = ''.join(
    f'<div class="fila-barra"><span>{nombre}</span>'
    f'<div class="pista centro"><span class="{"pos" if r > 0 else "neg"}" style="width:{abs(r) / 0.5 * 50:.0f}%"></span></div>'
    f'<span class="num">{r:+.2f}</span></div>'
    for nombre, r in CORRELACIONES)

st.markdown(f"""
<div id="modelo" class="titulo-seccion">
<div class="eyebrow">Evaluación</div>
<h2>Desempeño del modelo</h2>
<p>Validación cruzada estratificada de 10 folds sobre los {miles(N_VINOS)} vinos. Promedio de los folds de prueba.</p>
</div>
<div class="banda">
<div><b>{METRICAS['f1_test']:.3f}</b><span>F1 macro en prueba</span><small>promedio de ambas clases</small></div>
<div><b>{METRICAS['f1_train']:.3f}</b><span>F1 macro en entrenamiento</span><small>mismos 10 folds</small></div>
<div><b>+{METRICAS['f1_test'] - METRICAS['f1_vc']:.3f}</b><span>Mejora con GridSearch</span><small>{METRICAS['f1_vc']:.3f} antes de ajustar</small></div>
<div><b>{brecha:.1f} pp</b><span>Brecha train–test (F1)</span><small>{"sin señales de sobreajuste" if brecha < 5 else "posible sobreajuste"}</small></div>
</div>
<div class="graficos">
<div class="grafico"><h3>Comparación de modelos</h3><div class="nota">F1 macro en prueba, mismos 10 folds. Red neuronal tras GridSearch</div>{barras_modelos}</div>
<div class="grafico"><h3>Variables más asociadas a la calidad</h3><div class="nota">Correlación de Pearson con calidad Buena</div>{barras_corr}</div>
</div>
<div class="aviso">Limitaciones: todos los vinos son de una sola denominación de origen (Vinho Verde, Portugal), la nota proviene
de catas sensoriales subjetivas y la probabilidad de la red no está calibrada, por lo que debe leerse como una puntuación relativa.</div>
<div class="pie">{NOMBRE_APP} - Práctica 3, Analítica de Datos</div>
""", unsafe_allow_html=True)

# Verificación

In [ ]:
# Verificación de la predicción para la muestra capturada
# En el notebook se ve la salida; en Streamlit el print va a la consola, no a la página
print(data_preparada)
print('Predicción:', labelencoder.inverse_transform(modelo.predict(data_preparada))[0])